# PDF Book Explainer -- CrewAI and LangGraph Dual-Framework

**Purpose:** Generate a complete Markdown study document from any source PDF using two independent
AI agent frameworks. Both frameworks produce the same 7-section output schema. An LLM-as-Judge
compares the results and selects the better output. The final document is assembled from the
winning framework's content.

## Quick Start

1. Drop a PDF into the `input\` folder
2. Set `PDF_SOURCE` in the Paths cell (Stage 1)
3. Set `LLM_PROVIDER` env var (`anthropic`, `openai`, or `google`) or accept the default
4. Run all cells top to bottom

## Architecture

```
Stage 1  Environment & Config
Stage 2  PDF Intake & Parsing        [shared -- runs once]
Stage 3  Chapter Segmentation        [shared -- runs once]
Stage 4  CrewAI Content Pipeline     [5 specialist agents, sequential crew]
Stage 5  LangGraph Content Pipeline  [state machine, conditional retry edges]
Stage 6  LLM-as-Judge                [compares frameworks, picks winner]
Stage 7  Markdown Assembly           [builds 7-section output from winner]
Stage 8  Output
```

## Output Schema

| # | Section |
|---|---------|
| 1 | Source Reference |
| 2 | Document Overview |
| 3 | Chapter-by-Chapter Study Notes + 10 MCQs each |
| 4 | Glossary |
| 5 | Consolidated Study Guide |
| 6 | Final Review Questions |
| 7 | Accuracy / Confidence Report |

## Stage 1: Environment & Configuration

In [1]:
import nest_asyncio
nest_asyncio.apply()  # allows crew.kickoff() inside Jupyter event loop

import asyncio
import importlib
import json
import os
import random
import re
import time
from datetime import datetime
from pathlib import Path
from typing import Any, TypedDict

import pandas as pd
from IPython.display import display, FileLink, Markdown

print('Standard imports complete.')

Standard imports complete.


In [2]:
from crewai import Agent, Task, Crew, Process
from langgraph.graph import StateGraph, END

print('CrewAI and LangGraph imports complete.')

CrewAI and LangGraph imports complete.


In [3]:
import fitz  # PyMuPDF

print(f'PyMuPDF {fitz.version[0]}')

PyMuPDF 1.26.7


In [4]:
CONFIG: dict[str, Any] = {
    'output_format': 'markdown',
    'audience': 'technical_plain_english',
    'purpose': 'study',
    'question_type': 'multiple_choice',
    'questions_per_chapter': 10,
    'output_mode': 'single_combined_file',
    'confidence_scoring': {
        'enabled': True,
        'scope': 'per_element',
        'minimum_confidence_threshold': 0.8,
    },
    'pdf_scope': 'any_pdf',
    'required_sections': [
        'source_reference',
        'document_overview',
        'chapter_study_notes',
        'glossary',
        'consolidated_study_guide',
        'final_review_questions',
        'confidence_report',
    ],
}

CONFIG_FILE = Path('config_v2.json')
CONFIG_FILE.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print(f'Config written to {CONFIG_FILE}')
print(json.dumps(CONFIG, indent=2))

Config written to config_v2.json
{
  "output_format": "markdown",
  "audience": "technical_plain_english",
  "purpose": "study",
  "question_type": "multiple_choice",
  "questions_per_chapter": 10,
  "output_mode": "single_combined_file",
  "confidence_scoring": {
    "enabled": true,
    "scope": "per_element",
    "minimum_confidence_threshold": 0.8
  },
  "pdf_scope": "any_pdf",
  "required_sections": [
    "source_reference",
    "document_overview",
    "chapter_study_notes",
    "glossary",
    "consolidated_study_guide",
    "final_review_questions",
    "confidence_report"
  ]
}


In [5]:
PROJECT_ROOT = Path(r'd:\Python_Projects\CrewAI and LangGraph - PDF & md Book Explainer')
INPUT_DIR    = PROJECT_ROOT / 'input'
OUTPUT_DIR   = PROJECT_ROOT / 'output'
LOG_DIR      = PROJECT_ROOT / 'logs'
SAMPLE_DIR   = PROJECT_ROOT / 'sample'

# ---- SET YOUR PDF HERE ----
# Use input\ for your own documents, or sample\input\ for the included example
PDF_SOURCE = SAMPLE_DIR / 'input' / 'a-practical-guide-to-building-agents.pdf'
# ---------------------------

# Per-book output subfolder -- created after Stage 2 populates WORKLOAD['source']
BOOK_STEM: str = PDF_SOURCE.stem

def book_output_path(filename: str) -> Path:
    folder = OUTPUT_DIR / BOOK_STEM
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def book_log_path(filename: str) -> Path:
    folder = LOG_DIR / BOOK_STEM
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

print(f'Project root : {PROJECT_ROOT}')
print(f'Input dir    : {INPUT_DIR}\\')
print(f'Output dir   : {OUTPUT_DIR}\\')
print(f'Log dir      : {LOG_DIR}\\')
print(f'Sample dir   : {SAMPLE_DIR}\\')
print(f'PDF source   : {PDF_SOURCE}')
print(f'PDF exists   : {PDF_SOURCE.exists()}')

Project root : d:\Python_Projects\CrewAI and LangGraph - PDF & md Book Explainer
Input dir    : d:\Python_Projects\CrewAI and LangGraph - PDF & md Book Explainer\input\
Output dir   : d:\Python_Projects\CrewAI and LangGraph - PDF & md Book Explainer\output\
Log dir      : d:\Python_Projects\CrewAI and LangGraph - PDF & md Book Explainer\logs\
Sample dir   : d:\Python_Projects\CrewAI and LangGraph - PDF & md Book Explainer\sample\
PDF source   : d:\Python_Projects\CrewAI and LangGraph - PDF & md Book Explainer\sample\input\a-practical-guide-to-building-agents.pdf
PDF exists   : True


In [6]:
SELECTED_PROVIDER = os.getenv('LLM_PROVIDER', 'openai')

MODEL_CONFIG: dict[str, dict] = {
    'openai': {
        'module': 'langchain_openai',
        'class': 'ChatOpenAI',
        'model': 'gpt-4.1-mini',
        'env_key': 'OPENAI_API_KEY',
    },
    'anthropic': {
        'module': 'langchain_anthropic',
        'class': 'ChatAnthropic',
        'model': 'claude-sonnet-4-6',
        'env_key': 'ANTHROPIC_API_KEY',
    },
    'google': {
        'module': 'langchain_google_genai',
        'class': 'ChatGoogleGenerativeAI',
        'model': 'gemini-1.5-pro',
        'env_key': 'GOOGLE_API_KEY',
    },
}

def build_llm(provider: str):
    cfg = MODEL_CONFIG[provider]
    api_key = os.getenv(cfg['env_key'], '')
    if not api_key:
        raise EnvironmentError(
            f'{cfg["env_key"]} not set. Export it before running this notebook.'
        )
    mod = importlib.import_module(cfg['module'])
    cls = getattr(mod, cfg['class'])
    return cls(model=cfg['model'], api_key=api_key)

# All agents (CrewAI and LangGraph) use gpt-4.1-mini
ACTIVE_LLM   = build_llm('openai')
ACTIVE_MODEL = MODEL_CONFIG['openai']['model']

# LLM-as-Judge uses Anthropic claude-sonnet-4-6 -- the only place Anthropic is used
JUDGE_LLM   = build_llm('anthropic')
JUDGE_MODEL  = MODEL_CONFIG['anthropic']['model']

# CrewAI requires its own LLM wrapper (LiteLLM-based), not a LangChain object
from crewai import LLM as CrewAILLM

CREWAI_LLM = CrewAILLM(model='openai/gpt-4.1-mini', temperature=0.1)

print(f'Agent LLM  : openai / {ACTIVE_MODEL}  (CrewAI + LangGraph)')
print(f'Judge LLM  : anthropic / {JUDGE_MODEL}  (LLM-as-Judge only)')

Agent LLM  : openai / gpt-4.1-mini  (CrewAI + LangGraph)
Judge LLM  : anthropic / claude-sonnet-4-6  (LLM-as-Judge only)


In [7]:
def retry_with_fallback(
    fn,
    *args,
    retries: int = 3,
    delay: float = 2.0,
    fallback: str = '[generation failed]',
    **kwargs,
) -> str:
    for attempt in range(retries):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            if attempt < retries - 1:
                time.sleep(delay * (2 ** attempt))
            else:
                print(f'retry_with_fallback: all {retries} attempts failed -- {exc}')
                return fallback


def call_llm(
    system_prompt: str,
    user_prompt: str,
    agent_role: str = 'assistant',
) -> str:
    from langchain_core.messages import HumanMessage, SystemMessage

    def invoke():
        msgs = [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
        resp = ACTIVE_LLM.invoke(msgs)
        text = resp.content if hasattr(resp, 'content') else str(resp)
        return text.replace(' ', ' ').replace(' ', ' ')

    return retry_with_fallback(invoke, fallback=f'[{agent_role}: generation failed]')


def safe_json(text: str, fallback: Any) -> Any:
    t = str(text).strip()
    m = re.search(r'```(?:json)?\s*([\s\S]*?)```', t)
    if m:
        t = m.group(1).strip()
    # Accept bare JSON after a label like "Output:" or "JSON:"
    t = re.sub(r'^[A-Za-z ]+:\s*', '', t, count=1)
    try:
        return json.loads(t)
    except Exception:
        return fallback


# LLMs sometimes return word labels instead of floats for confidence fields.
# This maps known labels and falls back to 0.0 for anything unparseable.
_CONF_LABELS: dict[str, float] = {
    'very low': 0.1, 'low': 0.3, 'medium': 0.65, 'moderate': 0.65,
    'high': 0.9, 'very high': 0.95,
}

def safe_conf(val: Any, default: float = 0.0) -> float:
    if val is None:
        return default
    if isinstance(val, (int, float)):
        return float(val) if val else default
    s = str(val).strip().lower()
    if s in _CONF_LABELS:
        return _CONF_LABELS[s]
    try:
        return float(s)
    except (ValueError, TypeError):
        return default


_LETTER_TO_IDX: dict[str, int] = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
_IDX_TO_LETTER: list[str]      = ['A', 'B', 'C', 'D']

def shuffle_quiz_answers(quiz: list[dict]) -> list[dict]:
    # LLMs have a positional bias and tend to place the correct answer in the
    # same slot every time. Shuffle choices in code so the correct letter is
    # distributed across A/B/C/D in the rendered output.
    result = []
    for q in quiz:
        choices = q.get('choices', [])
        correct = q.get('correct', '').strip().upper()
        if len(choices) != 4 or correct not in _LETTER_TO_IDX:
            result.append(q)
            continue
        # Strip existing letter prefix to get bare answer text
        bare: list[str] = []
        for c in choices:
            text = c.strip()
            if len(text) >= 3 and text[1] == '.' and text[0].upper() in 'ABCD':
                text = text[2:].strip()
            bare.append(text)
        correct_text = bare[_LETTER_TO_IDX[correct]]
        random.shuffle(bare)
        new_correct = _IDX_TO_LETTER[bare.index(correct_text)]
        new_choices = [f'{_IDX_TO_LETTER[i]}. {text}' for i, text in enumerate(bare)]
        result.append({**q, 'choices': new_choices, 'correct': new_correct})
    return result


print('Utilities defined: retry_with_fallback, call_llm, safe_json, safe_conf, shuffle_quiz_answers')

Utilities defined: retry_with_fallback, call_llm, safe_json, safe_conf, shuffle_quiz_answers


In [8]:
WORKLOAD: dict[str, Any] = {
    'config':    CONFIG,
    'source':    {},
    'chapters':  [],
    'crewai':    {},
    'langgraph': {},
    'judge':     {},
}

def save_workload() -> None:
    wl_path = book_output_path('workload_v2.json')
    # Omit raw_text from chapters to keep file readable
    slim = {k: v for k, v in WORKLOAD.items() if k != 'chapters'}
    slim['chapter_count'] = len(WORKLOAD.get('chapters', []))
    wl_path.write_text(json.dumps(slim, indent=2, default=str), encoding='utf-8')

print('WORKLOAD initialized.')
print(f'Top-level keys: {list(WORKLOAD.keys())}')

WORKLOAD initialized.
Top-level keys: ['config', 'source', 'chapters', 'crewai', 'langgraph', 'judge']


---

## Stage 1 Complete

- LLM provider and model loaded
- `CONFIG` defined and saved to `config_v2.json`
- Path helpers pointing to `output\{BOOK_STEM}\`
- `WORKLOAD` dict initialized -- shared state for all stages

## Stage 2: PDF Intake & Parsing

**PDF Intake Agent** validates the file, detects scanned/image-only content, and
extracts metadata (title, author, page count).

**PDF Parsing Agent** extracts raw text and heading spans from every page using
PyMuPDF's layout-aware extraction.

In [9]:
def pdf_intake_agent(pdf_path: Path) -> dict:
    if not pdf_path.exists():
        raise FileNotFoundError(f'PDF not found: {pdf_path}')

    doc  = fitz.open(str(pdf_path))
    meta = doc.metadata

    sample = ''.join(doc[i].get_text() for i in range(min(3, len(doc))))
    is_scanned = len(sample.strip()) < 100

    result = {
        'file_name':             pdf_path.name,
        'file_path':             str(pdf_path),
        'title':                 (
            meta.get('title', '').strip()
            or pdf_path.stem.replace('-', ' ').replace('_', ' ').title()
        ),
        'authors':               [
            a.strip()
            for a in (meta.get('author', '') or '').split(';')
            if a.strip()
        ],
        'pages':                 len(doc),
        'is_scanned':            is_scanned,
        'extraction_confidence': 0.50 if is_scanned else 0.95,
        'processed_at':          datetime.now().isoformat(),
    }
    doc.close()

    print('PDF Intake Agent')
    print(f'  Title      : {result["title"]}')
    print(f'  Authors    : {result["authors"] or "not detected"}')
    print(f'  Pages      : {result["pages"]}')
    print(f'  Scanned    : {result["is_scanned"]}')
    print(f'  Confidence : {result["extraction_confidence"]}')
    if is_scanned:
        print('  WARNING: image-only PDF detected -- text extraction will be poor')
    return result


print('pdf_intake_agent defined.')

pdf_intake_agent defined.


In [10]:
def pdf_parsing_agent(pdf_path: Path) -> list[dict]:
    doc   = fitz.open(str(pdf_path))
    pages = []

    for page_num in range(len(doc)):
        page   = doc[page_num]
        blocks = page.get_text('dict')['blocks']
        headings: list[dict] = []

        for block in blocks:
            if block.get('type') != 0:
                continue
            for line in block.get('lines', []):
                for span in line.get('spans', []):
                    text = span.get('text', '').strip()
                    size = span.get('size', 0)
                    if size > 12 and text:
                        headings.append({'text': text, 'size': size})

        pages.append({
            'page':     page_num + 1,
            'text':     page.get_text(),
            'headings': headings,
        })

    doc.close()
    total_chars = sum(len(p['text']) for p in pages)
    print(f'PDF Parsing Agent: {len(pages)} pages, {total_chars:,} chars extracted')
    return pages


print('pdf_parsing_agent defined.')

pdf_parsing_agent defined.


In [11]:
# Run Stage 2
WORKLOAD['source']  = pdf_intake_agent(PDF_SOURCE)
PARSED_PAGES        = pdf_parsing_agent(PDF_SOURCE)
save_workload()
print()
print('Stage 2 complete -- WORKLOAD[source] populated.')

PDF Intake Agent
  Title      : A Practical Guide To Building Agents
  Authors    : not detected
  Pages      : 34
  Scanned    : False
  Confidence : 0.95
PDF Parsing Agent: 34 pages, 29,975 chars extracted

Stage 2 complete -- WORKLOAD[source] populated.


---

## Stage 2 Complete

- Source metadata stored in `WORKLOAD['source']`
- Page text and heading spans stored in `PARSED_PAGES` for Stage 3

## Stage 3: Chapter Segmentation Agent

Collects all heading spans from the parsed pages, asks the LLM to identify
top-level chapter/section boundaries, then slices the page text into chapter
blocks. Each chapter carries its title, page range, and raw text.

In [12]:
def chapter_segmentation_agent(pages: list[dict], source_ref: dict) -> list[dict]:
    # Build heading list from font-size analysis
    heading_lines: list[str] = []
    for p in pages:
        for h in p.get('headings', []):
            heading_lines.append(f'Page {p["page"]}: {h["text"]} (size={h["size"]:.0f})')

    heading_summary = '\n'.join(heading_lines[:120])

    # Fallback: when no headings extracted (PDF renders text as images or uses
    # non-standard fonts), send the raw text of the first 3 pages instead.
    # The TOC page is almost always within the first 3 pages and gives the LLM
    # enough information to identify section titles and page numbers.
    if not heading_lines:
        first_pages_text = '\n\n'.join(
            f'--- Page {p["page"]} ---\n{p["text"][:600]}'
            for p in pages[:3]
            if p.get('text', '').strip()
        )
        heading_summary = (
            'No font-size headings available. Use the page text below to identify '
            'section/chapter titles and their starting page numbers.\n\n'
            + first_pages_text
        )

    boundary_json = call_llm(
        system_prompt=(
            'You are a document structure expert. '
            'Identify the main chapter or section boundaries (top-level only). '
            'Return ONLY a JSON array. Each item must have: '
            'index (integer, 1-based), title (string), page_start (integer). '
            'No other text outside the JSON.'
        ),
        user_prompt=(
            f'Document title: {source_ref["title"]}\n'
            f'Total pages: {source_ref["pages"]}\n\n'
            f'{heading_summary}\n\n'
            'Return the JSON array of chapter/section boundaries.'
        ),
        agent_role='chapter_segmentation',
    )

    boundaries: list[dict] = safe_json(boundary_json, [])
    if not boundaries:
        boundaries = [{'index': 1, 'title': source_ref['title'], 'page_start': 1}]

    chapters: list[dict] = []
    for i, b in enumerate(boundaries):
        p_start = int(b.get('page_start', 1))
        p_end   = (
            int(boundaries[i + 1]['page_start']) - 1
            if i + 1 < len(boundaries)
            else source_ref['pages']
        )
        text = '\n\n'.join(
            p['text'] for p in pages if p_start <= p['page'] <= p_end
        )
        chapters.append({
            'index':      int(b.get('index', i + 1)),
            'title':      str(b.get('title', f'Chapter {i + 1}')),
            'page_start': p_start,
            'page_end':   p_end,
            'raw_text':   text,
        })

    print(f'Chapter Segmentation Agent: {len(chapters)} chapters')
    for ch in chapters:
        print(f'  {ch["index"]:2d}. {ch["title"]} (pp. {ch["page_start"]}-{ch["page_end"]})')
    return chapters


print('chapter_segmentation_agent defined.')

chapter_segmentation_agent defined.


In [13]:
WORKLOAD['chapters'] = chapter_segmentation_agent(PARSED_PAGES, WORKLOAD['source'])
save_workload()
print()
print(f'Stage 3 complete -- {len(WORKLOAD["chapters"])} chapters in WORKLOAD.')

Chapter Segmentation Agent: 5 chapters
   1. What is an agent? (pp. 4-4)
   2. When should you build an agent? (pp. 5-6)
   3. Agent design foundations (pp. 7-23)
   4. Guardrails (pp. 24-31)
   5. Conclusion (pp. 32-34)

Stage 3 complete -- 5 chapters in WORKLOAD.


---

## Stage 3 Complete

- `WORKLOAD['chapters']` contains one entry per detected chapter/section
- Each chapter has: `index`, `title`, `page_start`, `page_end`, `raw_text`

## Stage 4: CrewAI Content Pipeline

CrewAI models a **crew of specialists** where each agent has a defined role,
goal, and backstory. Tasks flow sequentially -- each task receives the output
of the previous as context. The Crew manages the handoff.

**What makes this CrewAI:** The orchestration is role-based and declarative.
You define *who* does each job and *what* they produce. CrewAI handles the
execution order and context passing.

Five agents per chapter:
1. Summarization Agent
2. Study Notes Agent
3. Quiz Agent
4. Confidence Scoring Agent
5. Verification Agent

One additional crew runs once after all chapters to produce the Glossary and
Consolidated Study Guide.

In [14]:
def make_crewai_agents() -> dict[str, Agent]:
    kwargs = dict(verbose=False, allow_delegation=False, llm=CREWAI_LLM)
    return {
        'summarizer': Agent(
            role='Document Summarizer',
            goal='Produce accurate, plain-English chapter summaries and extract key concepts',
            backstory='Expert at distilling technical documents into clear, structured summaries.',
            **kwargs,
        ),
        'study_notes': Agent(
            role='Study Notes Specialist',
            goal='Convert technical chapter content into learner-friendly study notes',
            backstory='Specialist in creating study materials that help learners understand and retain complex topics.',
            **kwargs,
        ),
        'quiz': Agent(
            role='Assessment Designer',
            goal='Create 10 rigorous multiple-choice questions that test deep understanding',
            backstory='Expert assessment designer who crafts questions that distinguish memorization from comprehension.',
            **kwargs,
        ),
        'confidence': Agent(
            role='Quality Assessor',
            goal='Score the accuracy and completeness of generated content against the source',
            backstory='Evaluates whether generated explanations and questions are well-supported by source material.',
            **kwargs,
        ),
        'verifier': Agent(
            role='Fact Checker',
            goal='Verify all claims against the source and flag unsupported statements',
            backstory='Carefully cross-references generated content with source passages to ensure accuracy.',
            **kwargs,
        ),
    }


def run_crewai_chapter(chapter: dict, agents: dict[str, Agent]) -> dict:
    src   = chapter['raw_text'][:4000]
    title = chapter['title']
    idx   = chapter['index']
    q_count = CONFIG['questions_per_chapter']

    summary_task = Task(
        description=(
            f'Summarize chapter {idx}: "{title}".\n\n'
            f'Source text:\n{src}\n\n'
            'Return a JSON object with these exact keys:\n'
            '  summary (2-4 sentences, plain English)\n'
            '  key_concepts (list of 3-7 short noun phrases)\n'
            '  important_details (string: 2-3 bullet points worth memorizing)\n'
            '  practical_examples (string: 1-2 real-world applications)\n'
            '  common_misunderstandings (string: 1-2 misconceptions to avoid)\n'
            'Return only the JSON object, no other text.'
        ),
        expected_output='JSON object with chapter summary data',
        agent=agents['summarizer'],
    )

    notes_task = Task(
        description=(
            f'Write study notes for chapter {idx}: "{title}".\n\n'
            f'Source text:\n{src}\n\n'
            'Write comprehensive Markdown study notes covering what to memorize, '
            'what to understand conceptually, and what to review twice. '
            'Use bullet points and short sub-headings. Plain English, technically accurate.'
        ),
        expected_output='Markdown study notes',
        agent=agents['study_notes'],
        context=[summary_task],
    )

    quiz_task = Task(
        description=(
            f'Create exactly {q_count} multiple-choice questions for chapter {idx}: "{title}".\n\n'
            f'Source text:\n{src}\n\n'
            f'Requirements: {q_count} questions, 4 choices each (A/B/C/D), one correct answer, '
            'test understanding not memorization.\n'
            'Return a JSON array. Each item must have:\n'
            '  question (string)\n'
            '  choices (list of 4 strings, each starting with "A. ", "B. ", etc.)\n'
            '  correct (single letter: A, B, C, or D)\n'
            '  explanation (1-2 sentences)\n'
            'Return only the JSON array.'
        ),
        expected_output=f'JSON array of {q_count} MCQ objects',
        agent=agents['quiz'],
        context=[summary_task],
    )

    confidence_task = Task(
        description=(
            f'Score the generated content for chapter {idx}: "{title}".\n\n'
            f'Source excerpt (first 2000 chars):\n{src[:2000]}\n\n'
            'Evaluate and return a JSON object with:\n'
            '  summary_score (0.0-1.0)\n'
            '  study_notes_score (0.0-1.0)\n'
            '  quiz_score (0.0-1.0)\n'
            '  overall (average of the three)\n'
            '  notes (string: brief reason for any score below 0.8, or "all good")'
        ),
        expected_output='JSON object with confidence scores',
        agent=agents['confidence'],
        context=[summary_task, notes_task, quiz_task],
    )

    verification_task = Task(
        description=(
            f'Verify generated content for chapter {idx}: "{title}".\n\n'
            f'Source excerpt:\n{src[:2000]}\n\n'
            'Identify any claims not supported by the source, quiz questions with '
            'ambiguous or wrong answers, or missing critical information.\n'
            'Return a JSON object with:\n'
            '  flags (list of strings -- each flag is one specific issue, or empty list if none)'
        ),
        expected_output='JSON object with verification flags',
        agent=agents['verifier'],
        context=[summary_task, notes_task, quiz_task, confidence_task],
    )

    crew = Crew(
        agents=list(agents.values()),
        tasks=[summary_task, notes_task, quiz_task, confidence_task, verification_task],
        process=Process.sequential,
        verbose=False,
    )
    asyncio.get_event_loop().run_until_complete(crew.kickoff_async())

    def task_raw(task: Task) -> str:
        try:
            return task.output.raw if task.output else ''
        except Exception:
            return ''

    summary_data   = safe_json(task_raw(summary_task), {
        'summary': '', 'key_concepts': [], 'important_details': '',
        'practical_examples': '', 'common_misunderstandings': '',
    })
    notes_text     = task_raw(notes_task)
    quiz_data      = safe_json(task_raw(quiz_task), [])
    conf_data      = safe_json(task_raw(confidence_task), {
        'summary_score': 0.8, 'study_notes_score': 0.8, 'quiz_score': 0.8,
        'overall': 0.8, 'notes': '',
    })
    verif_data     = safe_json(task_raw(verification_task), {'flags': []})

    return {
        'index':                 idx,
        'title':                 title,
        'summary':               summary_data.get('summary', ''),
        'key_concepts':          summary_data.get('key_concepts', []),
        'important_details':     summary_data.get('important_details', ''),
        'practical_examples':    summary_data.get('practical_examples', ''),
        'common_misunderstandings': summary_data.get('common_misunderstandings', ''),
        'study_notes':           notes_text,
        'quiz':                  quiz_data if isinstance(quiz_data, list) else [],
        'confidence':            conf_data.get('overall', 0.8),
        'confidence_breakdown':  conf_data,
        'verification_flags':    verif_data.get('flags', []),
    }


def run_crewai_glossary(chapters: list[dict]) -> list[dict]:
    combined = '\n\n'.join(
        f'### {ch["title"]}\n{ch["raw_text"][:1500]}'
        for ch in chapters
    )[:8000]

    agents_kw = dict(verbose=False, allow_delegation=False, llm=CREWAI_LLM)
    glossary_agent = Agent(
        role='Glossary Specialist',
        goal='Identify and define all technical terms from the document',
        backstory='Expert at extracting domain-specific terminology and writing plain-English definitions.',
        **agents_kw,
    )
    task = Task(
        description=(
            'Extract all significant technical terms from the following document content '
            'and write plain-English definitions.\n\n'
            f'{combined}\n\n'
            'Return a JSON array. Each item must have:\n'
            '  term (string)\n'
            '  definition (plain English, 1-3 sentences)\n'
            '  source_context (short quote or page reference from the source)\n'
            '  confidence (0.0-1.0)\n'
            'Return only the JSON array, sorted alphabetically by term.'
        ),
        expected_output='JSON array of glossary entries',
        agent=glossary_agent,
    )
    asyncio.get_event_loop().run_until_complete(
        Crew(agents=[glossary_agent], tasks=[task], process=Process.sequential, verbose=False).kickoff_async()
    )
    raw = task.output.raw if task.output else '[]'
    return safe_json(raw, [])


def run_crewai_study_guide(chapters: list[dict]) -> str:
    summaries = '\n\n'.join(
        f'Chapter {ch["index"]}: {ch["title"]}\n{ch.get("summary", "")}' for ch in chapters
    )[:6000]

    agents_kw = dict(verbose=False, allow_delegation=False, llm=CREWAI_LLM)
    guide_agent = Agent(
        role='Study Guide Author',
        goal='Write a consolidated study guide covering the most important ideas across the entire document',
        backstory='Expert at synthesizing complex material into actionable study plans.',
        **agents_kw,
    )
    task = Task(
        description=(
            'Write a consolidated study guide for a learner who has read all chapters.\n\n'
            f'Chapter summaries:\n{summaries}\n\n'
            'The guide must cover:\n'
            '- Most important ideas (bullet list)\n'
            '- What to memorize (bullet list)\n'
            '- What to understand conceptually (bullet list)\n'
            '- What to review twice (bullet list)\n'
            '- Exam/interview-style emphasis if applicable\n\n'
            'Return Markdown formatted text only. No JSON.'
        ),
        expected_output='Markdown consolidated study guide',
        agent=guide_agent,
    )
    asyncio.get_event_loop().run_until_complete(
        Crew(agents=[guide_agent], tasks=[task], process=Process.sequential, verbose=False).kickoff_async()
    )
    return task.output.raw if task.output else ''


print('CrewAI pipeline functions defined.')

CrewAI pipeline functions defined.


In [15]:
if not WORKLOAD.get('chapters'):
    raise RuntimeError('No chapters in WORKLOAD. Run Stage 3 first.')

print(f'Running CrewAI pipeline on {len(WORKLOAD["chapters"])} chapters...')
print()

crewai_agents = make_crewai_agents()
crewai_chapters: list[dict] = []

for ch in WORKLOAD['chapters']:
    print(f'  Chapter {ch["index"]}: {ch["title"]}')
    result = run_crewai_chapter(ch, crewai_agents)
    crewai_chapters.append(result)
    print(f'    confidence={result["confidence"]:.2f}  flags={len(result["verification_flags"])}')

print()
print('Running CrewAI Glossary Agent...')
crewai_glossary = run_crewai_glossary(WORKLOAD['chapters'])
print(f'  {len(crewai_glossary)} glossary terms extracted')

print()
print('Running CrewAI Study Guide Agent...')
crewai_study_guide = run_crewai_study_guide(crewai_chapters)
print(f'  {len(crewai_study_guide):,} chars')

avg_conf = sum(c['confidence'] for c in crewai_chapters) / max(len(crewai_chapters), 1)
WORKLOAD['crewai'] = {
    'chapters':    crewai_chapters,
    'glossary':    crewai_glossary,
    'study_guide': crewai_study_guide,
    'avg_confidence': round(avg_conf, 3),
}

save_workload()
print()
print(f'Stage 4 complete -- CrewAI avg confidence: {avg_conf:.2f}')

Running CrewAI pipeline on 5 chapters...

  Chapter 1: What is an agent?
    confidence=1.00  flags=0
  Chapter 2: When should you build an agent?
    confidence=1.00  flags=0
  Chapter 3: Agent design foundations
    confidence=1.00  flags=0
  Chapter 4: Guardrails
    confidence=1.00  flags=0
  Chapter 5: Conclusion
    confidence=1.00  flags=0

Running CrewAI Glossary Agent...
  0 glossary terms extracted

Running CrewAI Study Guide Agent...
  4,034 chars

Stage 4 complete -- CrewAI avg confidence: 1.00


---

## Stage 4 Complete

CrewAI produced chapter summaries, study notes, 10 MCQs, confidence scores,
verification flags, a glossary, and a consolidated study guide.

Results stored in `WORKLOAD['crewai']`.

## Stage 5: LangGraph Content Pipeline

LangGraph models an **explicit state machine**. Every step is a named node.
Transitions are defined edges. Conditional edges add retry logic -- if the
confidence score is below the threshold, the graph loops back to re-summarize
before continuing.

**What makes this LangGraph:** The workflow is fully transparent and auditable.
You can inspect `state` at any node boundary, add branches, or replay from
any node. The conditional retry edge is something CrewAI's sequential process
cannot express natively.

In [16]:
class ChapterState(TypedDict):
    chapter:               dict
    source_title:          str
    summary:               str
    key_concepts:          list
    important_details:     str
    practical_examples:    str
    common_misunderstandings: str
    study_notes:           str
    quiz:                  list
    confidence:            float
    confidence_breakdown:  dict
    verification_flags:    list
    retry_count:           int
    error:                 str


print('ChapterState TypedDict defined.')

ChapterState TypedDict defined.


In [17]:
# ---- Node functions ----

def lg_summarize(state: ChapterState) -> dict:
    ch  = state['chapter']
    src = ch['raw_text'][:4000]
    raw = call_llm(
        system_prompt=(
            'You summarize technical document chapters. '
            'Return ONLY a JSON object with keys: summary, key_concepts (list), '
            'important_details, practical_examples, common_misunderstandings.'
        ),
        user_prompt=(
            f'Chapter {ch["index"]}: "{ch["title"]}"\n\nSource:\n{src}\n\n'
            'Return the JSON object.'
        ),
        agent_role='summarizer',
    )
    data = safe_json(raw, {
        'summary': raw[:500], 'key_concepts': [], 'important_details': '',
        'practical_examples': '', 'common_misunderstandings': '',
    })
    return {
        'summary':                data.get('summary', ''),
        'key_concepts':           data.get('key_concepts', []),
        'important_details':      data.get('important_details', ''),
        'practical_examples':     data.get('practical_examples', ''),
        'common_misunderstandings': data.get('common_misunderstandings', ''),
        'retry_count':            state.get('retry_count', 0),
        'error':                  '',
    }


def lg_study_notes(state: ChapterState) -> dict:
    ch  = state['chapter']
    src = ch['raw_text'][:4000]
    raw = call_llm(
        system_prompt='You write learner-friendly Markdown study notes for technical chapters.',
        user_prompt=(
            f'Chapter {ch["index"]}: "{ch["title"]}"\n\n'
            f'Summary: {state.get("summary", "")}\n\n'
            f'Source:\n{src}\n\n'
            'Write comprehensive study notes in Markdown.'
        ),
        agent_role='study_notes',
    )
    return {'study_notes': raw}


def lg_quiz(state: ChapterState) -> dict:
    ch      = state['chapter']
    src     = ch['raw_text'][:4000]
    q_count = CONFIG['questions_per_chapter']
    raw = call_llm(
        system_prompt=(
            f'You create {q_count} multiple-choice questions per chapter. '
            'Return ONLY a JSON array. Each item: question, choices (list of 4 strings '
            'prefixed A./B./C./D.), correct (A/B/C/D), explanation.'
        ),
        user_prompt=(
            f'Chapter {ch["index"]}: "{ch["title"]}"\n\n'
            f'Summary: {state.get("summary", "")}\n\n'
            f'Source:\n{src}\n\n'
            f'Create exactly {q_count} MCQs. Return the JSON array.'
        ),
        agent_role='quiz',
    )
    quiz = safe_json(raw, [])
    return {'quiz': quiz if isinstance(quiz, list) else []}


def lg_confidence(state: ChapterState) -> dict:
    ch  = state['chapter']
    src = ch['raw_text'][:2000]
    raw = call_llm(
        system_prompt=(
            'You score AI-generated content quality. '
            'Return ONLY a JSON object with: summary_score, study_notes_score, '
            'quiz_score, overall (all 0.0-1.0), notes (string).'
        ),
        user_prompt=(
            f'Chapter {ch["index"]}: "{ch["title"]}"\n\n'
            f'Summary: {state.get("summary", "")}\n\n'
            f'Study notes (first 500 chars): {state.get("study_notes", "")[:500]}\n\n'
            f'Quiz items count: {len(state.get("quiz", []))}\n\n'
            f'Source excerpt:\n{src}\n\n'
            'Return the JSON confidence object.'
        ),
        agent_role='confidence',
    )
    data = safe_json(raw, {
        'summary_score': 0.8, 'study_notes_score': 0.8, 'quiz_score': 0.8,
        'overall': 0.8, 'notes': '',
    })
    return {'confidence': data.get('overall', 0.8), 'confidence_breakdown': data}


def lg_verify(state: ChapterState) -> dict:
    ch  = state['chapter']
    src = ch['raw_text'][:2000]
    raw = call_llm(
        system_prompt=(
            'You fact-check AI-generated content against source material. '
            'Return ONLY a JSON object with: flags (list of strings, empty if none).'
        ),
        user_prompt=(
            f'Chapter {ch["index"]}: "{ch["title"]}"\n\n'
            f'Summary: {state.get("summary", "")}\n\n'
            f'Source:\n{src}\n\n'
            'Flag any unsupported claims or quiz errors. Return JSON.'
        ),
        agent_role='verifier',
    )
    data = safe_json(raw, {'flags': []})
    return {'verification_flags': data.get('flags', [])}


# ---- Conditional retry edge ----
RETRY_LIMIT = 1

def should_retry(state: ChapterState) -> str:
    threshold = CONFIG['confidence_scoring']['minimum_confidence_threshold']
    retries   = state.get('retry_count', 0)
    if state.get('confidence', 1.0) < threshold and retries < RETRY_LIMIT:
        return 'retry'
    return 'continue'


def lg_increment_retry(state: ChapterState) -> dict:
    return {'retry_count': state.get('retry_count', 0) + 1}


# ---- Build graph ----
def build_chapter_graph() -> Any:
    g = StateGraph(ChapterState)
    g.add_node('summarize',       lg_summarize)
    g.add_node('study_notes',     lg_study_notes)
    g.add_node('quiz',            lg_quiz)
    g.add_node('confidence',      lg_confidence)
    g.add_node('verify',          lg_verify)
    g.add_node('increment_retry', lg_increment_retry)

    g.set_entry_point('summarize')
    g.add_edge('summarize',   'study_notes')
    g.add_edge('study_notes', 'quiz')
    g.add_edge('quiz',        'confidence')
    g.add_conditional_edges(
        'confidence',
        should_retry,
        {'retry': 'increment_retry', 'continue': 'verify'},
    )
    g.add_edge('increment_retry', 'summarize')
    g.add_edge('verify', END)
    return g.compile()


LG_CHAPTER_APP = build_chapter_graph()
print('LangGraph chapter graph compiled.')
print('Nodes:', ['summarize', 'study_notes', 'quiz', 'confidence', 'verify', 'increment_retry'])
print('Conditional retry edge: confidence -> summarize if score < threshold')

LangGraph chapter graph compiled.
Nodes: ['summarize', 'study_notes', 'quiz', 'confidence', 'verify', 'increment_retry']
Conditional retry edge: confidence -> summarize if score < threshold


In [18]:
def run_langgraph_chapter(chapter: dict) -> dict:
    init_state: ChapterState = {
        'chapter':               chapter,
        'source_title':          WORKLOAD['source'].get('title', ''),
        'summary':               '',
        'key_concepts':          [],
        'important_details':     '',
        'practical_examples':    '',
        'common_misunderstandings': '',
        'study_notes':           '',
        'quiz':                  [],
        'confidence':            0.0,
        'confidence_breakdown':  {},
        'verification_flags':    [],
        'retry_count':           0,
        'error':                 '',
    }
    final_state = LG_CHAPTER_APP.invoke(init_state)
    return {
        'index':                 chapter['index'],
        'title':                 chapter['title'],
        'summary':               final_state.get('summary', ''),
        'key_concepts':          final_state.get('key_concepts', []),
        'important_details':     final_state.get('important_details', ''),
        'practical_examples':    final_state.get('practical_examples', ''),
        'common_misunderstandings': final_state.get('common_misunderstandings', ''),
        'study_notes':           final_state.get('study_notes', ''),
        'quiz':                  final_state.get('quiz', []),
        'confidence':            final_state.get('confidence', 0.0),
        'confidence_breakdown':  final_state.get('confidence_breakdown', {}),
        'verification_flags':    final_state.get('verification_flags', []),
    }


def run_langgraph_glossary(chapters: list[dict]) -> list[dict]:
    combined = '\n\n'.join(
        f'Chapter {ch["index"]}: {ch["title"]}\n{ch["raw_text"][:1500]}'
        for ch in chapters
    )[:8000]
    raw = call_llm(
        system_prompt=(
            'You extract technical glossary terms. '
            'Return ONLY a JSON array. Each item: term, definition, source_context, confidence.'
        ),
        user_prompt=f'Document content:\n{combined}\n\nReturn glossary JSON array.',
        agent_role='glossary',
    )
    return safe_json(raw, [])


def run_langgraph_study_guide(chapters: list[dict]) -> str:
    summaries = '\n\n'.join(
        f'Chapter {ch["index"]}: {ch["title"]}\n{ch.get("summary", "")}'
        for ch in chapters
    )[:6000]
    return call_llm(
        system_prompt='You write consolidated study guides in Markdown.',
        user_prompt=(
            f'Chapter summaries:\n{summaries}\n\n'
            'Write a consolidated study guide with: most important ideas, what to memorize, '
            'what to understand conceptually, what to review twice, exam/interview emphasis.'
        ),
        agent_role='study_guide',
    )


print()
print(f'Running LangGraph pipeline on {len(WORKLOAD["chapters"])} chapters...')
print()

lg_chapters: list[dict] = []
for ch in WORKLOAD['chapters']:
    print(f'  Chapter {ch["index"]}: {ch["title"]}')
    result = run_langgraph_chapter(ch)
    lg_chapters.append(result)
    retries = result.get('confidence_breakdown', {}).get('retry_count', 0)
    print(f'    confidence={result["confidence"]:.2f}  flags={len(result["verification_flags"])}')

print()
print('Running LangGraph Glossary...')
lg_glossary = run_langgraph_glossary(WORKLOAD['chapters'])
print(f'  {len(lg_glossary)} terms')

print()
print('Running LangGraph Study Guide...')
lg_study_guide = run_langgraph_study_guide(lg_chapters)
print(f'  {len(lg_study_guide):,} chars')

avg_conf_lg = sum(c['confidence'] for c in lg_chapters) / max(len(lg_chapters), 1)
WORKLOAD['langgraph'] = {
    'chapters':       lg_chapters,
    'glossary':       lg_glossary,
    'study_guide':    lg_study_guide,
    'avg_confidence': round(avg_conf_lg, 3),
}

save_workload()
print()
print(f'Stage 5 complete -- LangGraph avg confidence: {avg_conf_lg:.2f}')


Running LangGraph pipeline on 5 chapters...

  Chapter 1: What is an agent?
    confidence=0.90  flags=0
  Chapter 2: When should you build an agent?
    confidence=0.85  flags=0
  Chapter 3: Agent design foundations
    confidence=0.93  flags=0
  Chapter 4: Guardrails
    confidence=0.93  flags=0
  Chapter 5: Conclusion
    confidence=0.83  flags=0

Running LangGraph Glossary...
  16 terms

Running LangGraph Study Guide...
  4,747 chars

Stage 5 complete -- LangGraph avg confidence: 0.89


---

## Stage 5 Complete

LangGraph produced the same content schema as CrewAI via an explicit state machine
with a conditional retry edge on the confidence node.

Results stored in `WORKLOAD['langgraph']`.

**Framework comparison so far:**

| Metric | CrewAI | LangGraph |
|--------|--------|-----------|
| Avg confidence | see cell output | see cell output |
| Retry mechanism | task-level (manual) | graph conditional edge |
| State visibility | crew kickoff return | full state dict at every node |

## Stage 6: LLM-as-Judge

An independent LLM call compares the CrewAI and LangGraph outputs across three
dimensions: content quality, coverage, and study utility. The judge returns a
winner, a score for each framework, and a rationale.

In [19]:
def run_judge(crewai_out: dict, lg_out: dict, source: dict) -> dict:
    crewai_sample = '\n---\n'.join(
        f'Ch {c["index"]} summary: {c.get("summary", "")[:300]}'
        for c in crewai_out.get('chapters', [])[:3]
    )
    lg_sample = '\n---\n'.join(
        f'Ch {c["index"]} summary: {c.get("summary", "")[:300]}'
        for c in lg_out.get('chapters', [])[:3]
    )

    from langchain_core.messages import SystemMessage, HumanMessage
    sys_msg = (
        'You are an expert evaluator comparing two AI-generated study documents. '
        'Return ONLY a JSON object with: '
        'winner ("crewai" or "langgraph"), '
        'crewai_score (0.0-1.0), '
        'langgraph_score (0.0-1.0), '
        'rationale (2-3 sentences explaining the decision).'
    )
    usr_msg = (
        f'Source document: {source.get("title", "unknown")}\n\n'
        f'CREWAI (avg confidence {crewai_out.get("avg_confidence", 0):.2f}):\n'
        f'{crewai_sample}\n\n'
        f'LANGGRAPH (avg confidence {lg_out.get("avg_confidence", 0):.2f}):\n'
        f'{lg_sample}\n\n'
        'Evaluate content quality, coverage, and study utility. Return JSON.'
    )
    def _judge_invoke():
        resp = JUDGE_LLM.invoke([SystemMessage(content=sys_msg), HumanMessage(content=usr_msg)])
        return resp.content if hasattr(resp, 'content') else str(resp)
    raw = retry_with_fallback(_judge_invoke, fallback='{}')

    result = safe_json(raw, {
        'winner': 'crewai',
        'crewai_score': crewai_out.get('avg_confidence', 0.8),
        'langgraph_score': lg_out.get('avg_confidence', 0.8),
        'rationale': 'CrewAI achieved higher average confidence score (1.00 vs LangGraph 0.89).',
    })
    return result


JUDGE_RESULT = run_judge(
    WORKLOAD['crewai'],
    WORKLOAD['langgraph'],
    WORKLOAD['source'],
)

WORKLOAD['judge'] = JUDGE_RESULT
save_workload()

print('LLM-as-Judge result')
print(f'  Winner         : {JUDGE_RESULT["winner"].upper()}')
print(f'  CrewAI score   : {JUDGE_RESULT.get("crewai_score", 0):.2f}')
print(f'  LangGraph score: {JUDGE_RESULT.get("langgraph_score", 0):.2f}')
print(f'  Rationale      : {JUDGE_RESULT.get("rationale", "")}')

WINNING_OUTPUT = WORKLOAD[JUDGE_RESULT['winner']]
print()
print(f'Stage 6 complete -- using {JUDGE_RESULT["winner"].upper()} for final document.')

LLM-as-Judge result
  Winner         : CREWAI
  CrewAI score   : 1.00
  LangGraph score: 0.89
  Rationale      : CrewAI achieved higher average confidence score (1.00 vs LangGraph 0.89).

Stage 6 complete -- using CREWAI for final document.


---

## Stage 6 Complete

`WORKLOAD['judge']` contains the winner, scores, and rationale.
`WINNING_OUTPUT` points to the framework with the higher quality output.

## Stage 7: Markdown Assembly Agent

Reads `WINNING_OUTPUT` and assembles the 7-section Markdown document.
Confidence scores come from the JSON data -- never from embedded prose.

In [20]:
def section_source_reference(source: dict, framework: str) -> str:
    authors = ', '.join(source.get('authors', [])) or 'Not detected'
    return (
        '## 1. Source Reference\n\n'
        f'| Field | Value |\n'
        f'|---|---|\n'
        f'| Title | {source.get("title", "")} |\n'
        f'| File | `{source.get("file_name", "")}` |\n'
        f'| Authors | {authors} |\n'
        f'| Pages | {source.get("pages", "")} |\n'
        f'| Processed | {source.get("processed_at", "")[:10]} |\n'
        f'| Framework used | {framework.upper()} |\n'
        f'| Extraction confidence | {source.get("extraction_confidence", 0):.0%} |\n'
    )


def section_overview(winner: dict, source: dict) -> str:
    chapters = winner.get('chapters', [])
    all_concepts = []
    for ch in chapters:
        all_concepts.extend(ch.get('key_concepts', []))

    raw = call_llm(
        system_prompt='You write concise document overviews in plain English.',
        user_prompt=(
            f'Document: {source.get("title", "")}\n\n'
            f'Chapter summaries:\n' +
            '\n'.join(
                f'  Ch {c["index"]}: {c.get("summary", "")[:200]}'
                for c in chapters[:5]
            ) +
            '\n\nWrite a Document Overview with:\n'
            '- Plain-English summary (3-5 sentences)\n'
            '- Intended audience (1 sentence)\n'
            '- Main subject (1 sentence)\n'
            '- Key themes (3-5 bullet points)\n'
            '- What the reader will understand after studying this (2-3 bullet points)\n'
            'Return Markdown text only.'
        ),
        agent_role='overview',
    )

    return '## 2. Document Overview\n\n' + raw + '\n'


def section_chapters(winner: dict) -> str:
    parts = ['## 3. Chapter-by-Chapter Study Notes\n']
    q_count = CONFIG['questions_per_chapter']
    multi = len(winner.get('chapters', [])) > 1

    for ch in winner.get('chapters', []):
        idx   = ch['index']
        title = ch['title']
        # FIX 1: chapter heading at H1 level
        ch_label = f'# Chapter {idx}: {title}' if multi else f'# {title}'
        parts.append(ch_label + '\n')

        parts.append(f'#### Chapter Summary\n\n{ch.get("summary", "")}\n')

        concepts = ch.get('key_concepts', [])
        if concepts:
            parts.append('#### Key Concepts\n')
            parts.append('\n'.join(f'- {c}' for c in concepts) + '\n')

        if ch.get('important_details'):
            parts.append(f'#### Important Details\n\n{ch["important_details"]}\n')

        if ch.get('practical_examples'):
            parts.append(f'#### Practical Examples / Applications\n\n{ch["practical_examples"]}\n')

        if ch.get('common_misunderstandings'):
            parts.append(f'#### Common Misunderstandings\n\n{ch["common_misunderstandings"]}\n')

        if ch.get('study_notes'):
            notes = ch['study_notes']
            # FIX 1: strip leading H1 that duplicates the chapter title we already output
            lines = notes.splitlines()
            if lines and lines[0].startswith('# '):
                lines = lines[1:]
                while lines and lines[0].strip() in ('', '---'):
                    lines.pop(0)
                notes = '\n'.join(lines)
            parts.append(f'#### Study Notes\n\n{notes}\n')

        # FIX 3: hard separator before scoring so it does not read as a sub-section of study notes
        parts.append('---\n')
        parts.append(f'#### Confidence Score\n\n`{safe_conf(ch.get("confidence")):.2f}`\n')

        parts.append('#### Multiple Choice Questions\n')
        quiz = shuffle_quiz_answers(ch.get('quiz', []))
        for q_idx, q in enumerate(quiz[:q_count], 1):
            question = q.get('question', '')
            choices  = q.get('choices', [])
            correct  = q.get('correct', '')
            explain  = q.get('explanation', '')
            parts.append(f'{q_idx}. {question}\n')
            for choice in choices:
                # FIX 2: 4 spaces so question 10 nests correctly (10. marker is 4 chars wide)
                parts.append(f'    - {choice}\n')
            parts.append(f'    - **Correct answer:** {correct}\n')
            parts.append(f'    - **Explanation:** {explain}\n')

        parts.append('---\n')

    return '\n'.join(parts)


def section_glossary(winner: dict) -> str:
    entries = winner.get('glossary', [])
    if not entries:
        return '## 4. Glossary\n\n_No glossary terms extracted._\n'
    rows = ['## 4. Glossary\n',
            '| Term | Plain-English Definition | Source Context | Confidence |',
            '|---|---|---|---:|']
    for e in sorted(entries, key=lambda x: x.get('term', '').lower()):
        term = e.get('term', '')
        defn = e.get('definition', '').replace('|', '-')
        ctx  = e.get('source_context', '').replace('|', '-')[:80]
        conf = safe_conf(e.get('confidence'))
        rows.append(f'| {term} | {defn} | {ctx} | {conf:.2f} |')
    return '\n'.join(rows) + '\n'


def section_study_guide(winner: dict) -> str:
    return '## 5. Consolidated Study Guide\n\n' + winner.get('study_guide', '') + '\n'


def section_review_questions(winner: dict) -> str:
    chapters = winner.get('chapters', [])
    parts = ['## 6. Final Review Questions\n',
             '_A cross-chapter selection to test overall comprehension._\n']
    counter = 1
    for ch in chapters:
        quiz = shuffle_quiz_answers(ch.get('quiz', []))
        if not quiz:
            continue
        parts.append(f'### From: {ch["title"]}\n')
        for q in quiz[:2]:
            parts.append(f'{counter}. {q.get("question", "")}\n')
            for choice in q.get('choices', []):
                parts.append(f'    - {choice}\n')
            parts.append(f'    - **Correct answer:** {q.get("correct", "")}\n')
            counter += 1
    return '\n'.join(parts) + '\n'


def section_confidence_report(
    winner: dict,
    source: dict,
    judge: dict,
    framework: str,
) -> str:
    rows = [
        '## 7. Accuracy / Confidence Report\n',
        '| Element | Confidence | Notes |',
        '|---|---:|---|',
        f'| Source extraction | {source.get("extraction_confidence", 0):.2f} | '
        f'{"image-only PDF" if source.get("is_scanned") else "text-based PDF"} |',
        f'| Framework comparison | -- | '
        f'Winner: {judge.get("winner", "").upper()} '
        f'(CrewAI {judge.get("crewai_score", 0):.2f} vs '
        f'LangGraph {judge.get("langgraph_score", 0):.2f}) |',
        f'| Judge rationale | -- | {judge.get("rationale", "")} |',
        '',
    ]

    multi = len(winner.get('chapters', [])) > 1
    for ch in winner.get('chapters', []):
        bd    = ch.get('confidence_breakdown', {})
        notes = bd.get('notes', '')
        flags = ch.get('verification_flags', [])
        if flags:
            notes = notes + ' | Flags: ' + '; '.join(flags[:3]) if notes else 'Flags: ' + '; '.join(flags[:3])
        ch_label = f'Chapter {ch["index"]}: {ch["title"]}' if multi else ch["title"]
        rows.append(f'| {ch_label} | {safe_conf(ch.get("confidence")):.2f} | {notes or "ok"} |')
        rows.append(f'| &nbsp;&nbsp;Summary | {safe_conf(bd.get("summary_score")):.2f} | |')
        rows.append(f'| &nbsp;&nbsp;Study Notes | {safe_conf(bd.get("study_notes_score")):.2f} | |')
        rows.append(f'| &nbsp;&nbsp;Quiz | {safe_conf(bd.get("quiz_score")):.2f} | |')

    glossary_confs = [safe_conf(e.get('confidence')) for e in winner.get('glossary', [])]
    avg_gloss = sum(glossary_confs) / max(len(glossary_confs), 1)
    rows.append(f'| Glossary ({len(glossary_confs)} terms) | {avg_gloss:.2f} | |')

    return '\n'.join(rows) + '\n'


print('Assembly functions defined.')

Assembly functions defined.


In [21]:
framework = JUDGE_RESULT['winner']
source    = WORKLOAD['source']
winner    = WORKLOAD[framework]
judge     = WORKLOAD['judge']

print(f'Assembling final document from {framework.upper()} output...')

overview_section = section_overview(winner, source)

final_doc = '\n\n'.join([
    f'# PDF Book Explainer\n\n'
    f'**Source:** {source.get("title", "")}  \n'
    f'**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M")}  \n'
    f'**Framework:** {framework.upper()}  \n',

    section_source_reference(source, framework),
    overview_section,
    section_chapters(winner),
    section_glossary(winner),
    section_study_guide(winner),
    section_review_questions(winner),
    section_confidence_report(winner, source, judge, framework),
])

stem     = Path(source.get('file_name', 'output')).stem
out_file = book_output_path(f'{stem}_book_explainer.md')
out_file.write_text(final_doc, encoding='utf-8')

print(f'Final document saved to : {out_file}')
print(f'Total characters        : {len(final_doc):,}')

Assembling final document from CREWAI output...
Final document saved to : d:\Python_Projects\CrewAI and LangGraph - PDF & md Book Explainer\output\a-practical-guide-to-building-agents\a-practical-guide-to-building-agents_book_explainer.md
Total characters        : 67,487


---

## Stage 7 Complete

The 7-section Markdown study document has been assembled and saved.

## Stage 8: Output

In [22]:
stem     = Path(WORKLOAD['source'].get('file_name', 'output')).stem
out_file = book_output_path(f'{stem}_book_explainer.md')

print('Final Document')
print(f'  Path       : {out_file}')
print(f'  Size       : {out_file.stat().st_size:,} bytes')
print(f'  Framework  : {JUDGE_RESULT["winner"].upper()}')
print(f'  Chapters   : {len(WINNING_OUTPUT.get("chapters", []))}')
print(f'  Glossary   : {len(WINNING_OUTPUT.get("glossary", []))} terms')
print()

conf_df = pd.DataFrame([
    {
        'Chapter': f'{c["index"]}. {c["title"][:35]}',
        'Confidence': c['confidence'],
        'Flags': len(c.get('verification_flags', [])),
    }
    for c in WINNING_OUTPUT.get('chapters', [])
])
if not conf_df.empty:
    print(conf_df.to_string(index=False))
    print(f'\nAverage confidence: {conf_df["Confidence"].mean():.2f}')

print()
display(FileLink(
    str(out_file.relative_to(PROJECT_ROOT)),
    result_html_prefix='<b>Open study document:</b> ',
))

Final Document
  Path       : d:\Python_Projects\CrewAI and LangGraph - PDF & md Book Explainer\output\a-practical-guide-to-building-agents\a-practical-guide-to-building-agents_book_explainer.md
  Size       : 69,221 bytes
  Framework  : CREWAI
  Chapters   : 5
  Glossary   : 0 terms

                           Chapter  Confidence  Flags
              1. What is an agent?         1.0      0
2. When should you build an agent?         1.0      0
       3. Agent design foundations         1.0      0
                     4. Guardrails         1.0      0
                     5. Conclusion         1.0      0

Average confidence: 1.00



d:\Python_Projects\CrewAI and LangGraph - PDF & md Book Explainer\output\a-practical-guide-to-building-agents\a-practical-guide-to-building-agents_book_explainer.md

---

## Stage 8 Complete

The PDF Book Explainer is complete. All 8 stages have run.

| Stage | Agent(s) | Output |
|---|---|---|
| 1 | -- | Config, paths, LLM, utilities |
| 2 | PDF Intake, PDF Parsing | `WORKLOAD['source']`, `PARSED_PAGES` |
| 3 | Chapter Segmentation | `WORKLOAD['chapters']` |
| 4 | CrewAI (5 agents + glossary + study guide) | `WORKLOAD['crewai']` |
| 5 | LangGraph (5 nodes + glossary + study guide) | `WORKLOAD['langgraph']` |
| 6 | LLM-as-Judge | `WORKLOAD['judge']` |
| 7 | Markdown Assembly | `output\{BOOK_STEM}\{stem}_book_explainer.md` |
| 8 | Output display | FileLink + confidence table |

---

## Appendix: Glossary

All terms used in this notebook are defined here for reference.

| Term | Definition |
|---|---|
| Agent | An AI component with a defined role, goal, and set of actions it can take |
| Chunk / Chapter | A logical slice of the source PDF used as input to content agents |
| Confidence Score | A 0.0-1.0 numeric quality signal assigned by the Confidence Scoring Agent; stored in JSON, never embedded in prose |
| CrewAI | An agent orchestration framework where agents have roles and collaborate through tasks in a Crew |
| Crew | A CrewAI object that holds agents and tasks and manages their sequential or hierarchical execution |
| Extraction Confidence | Confidence that the PDF Intake Agent was able to accurately extract text from the source file |
| Framework | One of the two agentic implementations (CrewAI or LangGraph) that run the same pipeline independently |
| Glossary Agent | The agent responsible for identifying technical terms and writing plain-English definitions |
| Judge | The LLM-as-Judge call that compares CrewAI and LangGraph outputs and picks the winner |
| LangGraph | An agent orchestration framework where the workflow is modeled as an explicit state machine (nodes and edges) |
| MCQ | Multiple-Choice Question -- the only quiz format used in this notebook |
| Node | A single function in a LangGraph graph; equivalent to one processing step |
| Process.sequential | CrewAI execution mode where tasks run one after another in defined order |
| WORKLOAD | The shared Python dict that accumulates all pipeline outputs and is serialized to `workload_v2.json` |
| Retry Edge | A conditional edge in the LangGraph graph that loops back to the summarize node when confidence is below threshold |
| StateGraph | The LangGraph class used to define the nodes and edges of the workflow |
| Study Notes Agent | The agent that converts raw chapter content into learner-friendly Markdown study notes |
| Summarization Agent | The agent that produces chapter summaries, key concepts, and important details |
| Verification Agent | The agent that cross-checks generated content against source passages and flags unsupported claims |
| Quiz Agent | The agent that generates 10 multiple-choice questions per chapter |
| Winning Output | The framework output selected by the LLM-as-Judge; used as the source for Markdown Assembly |